# Jaxpy Benchmark

Benchmark `segmented (log0, seg=8)` vs `pure GL` vs `base jampy` on the same inputs.

This notebook is designed to run on either CPU or GPU. Set `GPU = True` in the first cell before running on a CUDA-enabled server.

In [ ]:
# Backend and local-package setup
from pathlib import Path
import os
import sys

GPU = False
requested_label = "gpu" if GPU else "cpu"
os.environ["JAX_PLATFORMS"] = "cuda,cpu" if GPU else "cpu"

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import numpyro

try:
    active_backend = jax.default_backend()
    active_devices = jax.devices()
except Exception as exc:
    print(f"[backend] initialization failed: {exc}")
    print("[backend] Try GPU=False, or verify CUDA-enabled jaxlib in this environment.")
    raise

numpyro.set_platform("gpu" if active_backend == "cuda" else "cpu")

print(f"Requested backend: {requested_label}")
print("JAX_PLATFORMS:", os.environ.get("JAX_PLATFORMS"))
print("Active JAX backend:", active_backend)
print("JAX devices:", active_devices)

def find_local_jaxpy_root():
    cwd = Path.cwd()
    candidates = [
        cwd,
        cwd / "jaxpy",
        cwd / "jaxpy" / "jaxpy",
        cwd.parent,
        cwd.parent / "jaxpy",
        cwd.parent / "jaxpy" / "jaxpy",
    ]
    for candidate in candidates:
        if (candidate / "jam_axi_proj_jax.py").exists():
            return candidate
    raise RuntimeError("Cannot locate local jaxpy package from current working directory.")

local_jaxpy_root = str(find_local_jaxpy_root())
if local_jaxpy_root not in sys.path:
    sys.path.insert(0, local_jaxpy_root)

print("Using local jaxpy root:", local_jaxpy_root)

In [ ]:
# Common imports and benchmark input
import importlib
import json
import time
import matplotlib.pyplot as plt

import nquad as nq
import jam_axi_proj_jax as jam_mod

def make_inputs(nx_grid=50, ny_grid=50, xlim=40.0, ylim=40.0):
    x_lin = np.linspace(-xlim, xlim, nx_grid)
    y_lin = np.linspace(-ylim, ylim, ny_grid)
    x_mesh, y_mesh = np.meshgrid(x_lin, y_lin, indexing="xy")
    xbin = x_mesh.ravel()
    ybin = y_mesh.ravel()

    inc = 60.0
    r = jnp.sqrt(xbin**2 + (ybin / jnp.cos(jnp.radians(inc)))**2)
    a = 40.0
    vr = 2000.0 * jnp.sqrt(r) / (r + a)
    vel = vr * jnp.sin(jnp.radians(inc)) * xbin / r
    sig = 8700.0 / (r + a)
    rms = jnp.sqrt(vel**2 + sig**2)

    surf = jnp.array([39483., 37158., 30646., 17759., 5955.1, 1203.5, 174.36, 21.105, 2.3599, 0.25493])
    sigma = jnp.array([0.153, 0.515, 1.58, 4.22, 10., 22.4, 48.8, 105., 227., 525.])
    qobs = jnp.full_like(sigma, 0.57)

    distance = 16.5
    mbh = 1e8
    beta = jnp.full_like(surf, 0.2)
    sigmapsf = jnp.array([0.6, 1.2])
    normpsf = jnp.array([0.7, 0.3])
    pixsize = 0.8
    goodbins = jnp.ones_like(r, dtype=bool)

    return dict(
        surf_lum=surf,
        sigma_lum=sigma,
        qobs_lum=qobs,
        surf_pot=surf,
        sigma_pot=sigma,
        qobs_pot=qobs,
        inc=inc,
        mbh=mbh,
        distance=distance,
        xbin=xbin,
        ybin=ybin,
        data=rms,
        sigmapsf=sigmapsf,
        normpsf=normpsf,
        beta=beta,
        pixsize=pixsize,
        logistic=False,
        moment="zz",
        goodbins=goodbins,
        align="sph",
        ml=None,
        nrad=20,
        nang=10,
        nlos=1500,
        step=0.05,
        quiet=True,
    )

common_kwargs = make_inputs()
print(f"Grid: 50x50 -> {common_kwargs['xbin'].size} bins")
print("align='sph', nrad=20, nang=10, nlos=1500, step=0.05")

In [ ]:
# Benchmark helpers
def sync_jax_model(output_tuple):
    model = output_tuple[0]
    model = jax.block_until_ready(model)
    return np.asarray(model)

def bench_jax(mode_name, repeat=5):
    global nq, jam_mod
    nq = importlib.reload(nq)
    jam_mod = importlib.reload(jam_mod)

    if mode_name == "segmented":
        nq.quad = nq.def_segmented_quad(n=20, segments=8, log_L=3.0)
        quad_label = "segmented (log0, seg=8)"
    elif mode_name == "pure_gl":
        nq.quad = nq.def_nth_order_quad(n=20)
        quad_label = "pure GL"
    else:
        raise ValueError(f"Unknown mode: {mode_name}")

    jam_obj = jam_mod.jam_axi_proj()

    t0 = time.perf_counter()
    out0 = jam_obj.get_kinematics(**common_kwargs)
    model0 = sync_jax_model(out0)
    t_first = time.perf_counter() - t0

    post_times = []
    post_means = []
    for _ in range(repeat):
        t1 = time.perf_counter()
        out = jam_obj.get_kinematics(**common_kwargs)
        model = sync_jax_model(out)
        post_times.append(time.perf_counter() - t1)
        post_means.append(float(np.mean(model)))

    return {
        "mode": quad_label,
        "kind": "jax",
        "first_s": float(t_first),
        "post_mean_s": float(np.mean(post_times)),
        "post_std_s": float(np.std(post_times)),
        "post_min_s": float(np.min(post_times)),
        "model_mean_first": float(np.mean(model0)),
        "model_mean_post": float(np.mean(post_means)),
    }

def load_base_jampy():
    cwd = Path.cwd()
    candidates = [
        cwd / "Jampy" / "jampy-8.1.4",
        cwd.parent / "Jampy" / "jampy-8.1.4",
        cwd.parent.parent / "Jampy" / "jampy-8.1.4",
        cwd.parent.parent.parent / "Jampy" / "jampy-8.1.4",
    ]
    local_jampy_root = next((p for p in candidates if p.exists()), None)
    if local_jampy_root is not None:
        local_jampy_root = str(local_jampy_root)
        if local_jampy_root not in sys.path:
            sys.path.insert(0, local_jampy_root)
        print(f"Using local jampy source: {local_jampy_root}")
    else:
        print("Local Jampy/jampy-8.1.4 not found; using installed jampy package.")

    import jampy
    import jampy.axi.jam_axi_proj as jam_base_mod
    jam_base_mod = importlib.reload(jam_base_mod)
    from jampy.axi.jam_axi_proj import jam_axi_proj as jam_base
    return jampy, jam_base

def bench_base(repeat=5):
    jampy, jam_base = load_base_jampy()
    scalar_keys = {"inc", "mbh", "distance", "pixsize", "step"}
    pass_keys = {"logistic", "moment", "align", "ml"}

    base_kwargs = {}
    for key, value in common_kwargs.items():
        if key == "quiet":
            continue
        if key in scalar_keys:
            base_kwargs[key] = float(value)
        elif key in pass_keys:
            base_kwargs[key] = value
        elif key == "goodbins":
            base_kwargs[key] = np.asarray(value, dtype=bool)
        else:
            base_kwargs[key] = np.asarray(value)

    t0 = time.perf_counter()
    jam0 = jam_base(**base_kwargs, plot=False, quiet=True)
    model0 = np.asarray(jam0.model)
    t_first = time.perf_counter() - t0

    post_times = []
    post_means = []
    for _ in range(repeat):
        t1 = time.perf_counter()
        jam_b = jam_base(**base_kwargs, plot=False, quiet=True)
        model = np.asarray(jam_b.model)
        post_times.append(time.perf_counter() - t1)
        post_means.append(float(np.mean(model)))

    return {
        "mode": f"base jampy {jampy.__version__}",
        "kind": "base",
        "first_s": float(t_first),
        "post_mean_s": float(np.mean(post_times)),
        "post_std_s": float(np.std(post_times)),
        "post_min_s": float(np.min(post_times)),
        "model_mean_first": float(np.mean(model0)),
        "model_mean_post": float(np.mean(post_means)),
    }

In [ ]:
# Run benchmark and save outputs
repeat_post = 5

results = [
    bench_jax("segmented", repeat=repeat_post),
    bench_jax("pure_gl", repeat=repeat_post),
    bench_base(repeat=repeat_post),
]

print("=== Benchmark Results ===")
for row in results:
    print(
        f"{row['mode']:>24s} | first={row['first_s']:.4f}s | "
        f"post_mean={row['post_mean_s']:.4f}s (+/-{row['post_std_s']:.4f}, min={row['post_min_s']:.4f}) | "
        f"mean(model): first={row['model_mean_first']:.6f}, post={row['model_mean_post']:.6f}"
    )

labels = [row['mode'] for row in results]
first_vals = [row['first_s'] for row in results]
post_vals = [row['post_mean_s'] for row in results]

x = np.arange(len(labels))
width = 0.36

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.bar(x - width / 2, first_vals, width=width, label='First call (compile/trace included)', color='#4C78A8')
ax.bar(x + width / 2, post_vals, width=width, label='Post-compile mean', color='#F58518')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=10, ha='right')
ax.set_ylabel('Runtime [s]')
ax.set_title(f'JAX segmented vs pure GL vs base jampy, align=sph ({active_backend})')
ax.legend()
ax.grid(alpha=0.25, axis='y')
fig.tight_layout()

plot_path = 'jax_runtime_benchmark_sph.png'
json_path = 'jax_runtime_benchmark_sph.json'
fig.savefig(plot_path, dpi=220, bbox_inches='tight')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)

print(f'Saved plot: {plot_path}')
print(f'Saved JSON: {json_path}')
plt.show()